# Monte Carlo Shapley Value — Convergence Analysis

This notebook automatically discovers every hourly checkpoint file in the results folder,
compares each **consecutive pair** of checkpoints ($i$ vs $i{+}1$), and plots a loss-style
convergence curve showing how much the Shapley estimates are still changing over time.

Designed to be re-run at any point while the Monte Carlo experiment is still running in the
background — it will simply pick up however many checkpoint files currently exist.

## Cell 1 — Imports & Configuration

In [ ]:
from pathlib import Path
import re
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

# ── Aesthetics ──────────────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.dpi": 130,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})

# Linux Libertine — the body face of the ACM 'acmart' template used for SIGMOD,
# installed under ~/.fonts. The family has to be named explicitly because bare
# 'serif' resolves to DejaVu Serif. Sizes, weights and figsizes are unchanged.
plt.rc('font', family='serif', serif=['Linux Libertine'])
plt.rc('mathtext', fontset='custom', rm='Linux Libertine',
       it='Linux Libertine:italic', bf='Linux Libertine:bold')
plt.rc('pdf', fonttype=42)   # embed the real font instead of outlining glyphs

import matplotlib.font_manager as fm
import matplotlib.text as mtext

# Load the font from its file and attach it to each text artist, which no
# rcParams reset can undo. These are the regular and bold faces of Linux
# Libertine from the SIGMOD font package, installed in ~/.fonts.
PAPER_FONT_FILES = {
    'normal': str(Path.home() / '.fonts' / 'LinLibertine_Rah.ttf'),
    'bold':   str(Path.home() / '.fonts' / 'LinLibertine_RBah.ttf'),
}
for _f in PAPER_FONT_FILES.values():
    fm.fontManager.addfont(_f)


def apply_paper_font(fig):
    """Force the paper's font file onto every text artist in `fig`.

    Called once after a figure is fully built, this covers axis labels, ticks,
    titles, annotations and legend entries in one pass. Each artist keeps its
    own size and style; only the font file changes, chosen to match its weight.
    """
    for t in fig.findobj(mtext.Text):
        weight = t.get_fontweight()
        is_bold = weight in ('bold', 'heavy', 'semibold', 600, 700, 800, 900)
        prop = t.get_fontproperties().copy()
        prop.set_file(PAPER_FONT_FILES['bold' if is_bold else 'normal'])
        t.set_fontproperties(prop)

print("Imports OK.")

## Cell 2 — Discover & Load All Hourly Checkpoints

Scans the results folder for every file matching `monte_carlo_shapley_<H>.0h.csv`,
sorts them numerically by elapsed hour, and loads them all — feature column as index.

Re-run this cell any time to pick up newly written checkpoints from the still-running experiment.

In [ ]:
# ── Parameters — edit these for each run you want to analyze ───────────────
RESULTS = Path("/path/to/rel-<dataset>/<task>/<HGS|transformer>/monteCarlo_explainer/monte_carlo_results_<timestamp>")
DATA  = "<dataset>"   # short label for output filenames, e.g. 'trial', 'f1', 'arxiv'
TASK  = "<task>"      # short label, e.g. 'so' (study-outcome), 'dnf' (driver-dnf)
MODEL = "<model>"     # short label, e.g. 'relgt' or 'hgs'

OUTPUT_DIR = RESULTS.parent   # <dataset>/<task>/<model>/monte
OUTPUT_DIR.mkdir(exist_ok=True)

# ── Discover checkpoint files ────────────────────────────────────────────────
CHECKPOINT_PATTERN = re.compile(r"monte_carlo_shapley_([\d.]+)h\.csv$")

checkpoint_files = []
for f in RESULTS.glob("monte_carlo_shapley_*h.csv"):
    m = CHECKPOINT_PATTERN.search(f.name)
    if m:
        hour = float(m.group(1))
        checkpoint_files.append((hour, f))

# Sort numerically by elapsed hour (NOT alphabetically — avoids '10h' < '2h' bugs)
checkpoint_files.sort(key=lambda pair: pair[0])

if not checkpoint_files:
    raise FileNotFoundError(
        f"No checkpoint files matching 'monte_carlo_shapley_<H>.0h.csv' found in {RESULTS}"
    )

print(f"Found {len(checkpoint_files)} checkpoint(s):")
for hour, f in checkpoint_files:
    print(f"  {hour:>6.1f}h  ->  {f.name}")

# ── Load every checkpoint, feature as index ──────────────────────────────────
# checkpoints: list of (elapsed_hour, dataframe), in ascending hour order
checkpoints = [(hour, pd.read_csv(f, index_col="feature")) for hour, f in checkpoint_files]

print(f"\nLoaded {len(checkpoints)} checkpoints. Example (first one):")
display(checkpoints[0][1].head())

## Cell 3 — Consecutive-Pair Difference Calculation

For every step $i \to i{+}1$ we compute, per feature:

$$\text{Absolute Difference}_i = |\text{shapley\_value}_{i+1} - \text{shapley\_value}_{i}|$$

$$\text{Abs Absolute Difference}_i = |\text{abs\_shapley\_value}_{i+1} - \text{abs\_shapley\_value}_{i}|$$

In [ ]:
def consecutive_diff(hour1, df1, hour2, df2):
    """
    Compute per-feature absolute differences between two consecutive checkpoints.

    Returns a DataFrame indexed by feature with columns:
        'absolute_diff'      – |shapley_value(t2) - shapley_value(t1)|
        'abs_absolute_diff'  – |abs_shapley_value(t2) - abs_shapley_value(t1)|
    plus metadata about the step (hour1, hour2, label).
    """
    shared_idx = df1.index.intersection(df2.index)
    if len(shared_idx) < len(df1) or len(shared_idx) < len(df2):
        missing = set(df1.index).symmetric_difference(set(df2.index))
        print(f"[{hour1}h -> {hour2}h] Warning — {len(missing)} feature(s) not shared: {missing}")

    a = df1.loc[shared_idx]
    b = df2.loc[shared_idx]

    diff = pd.DataFrame(index=shared_idx)
    diff["absolute_diff"]     = (b["shapley_value"]     - a["shapley_value"]).abs()
    diff["abs_absolute_diff"] = (b["abs_shapley_value"] - a["abs_shapley_value"]).abs()
    diff["hour_from"] = hour1
    diff["hour_to"]   = hour2
    diff["step_label"] = f"{hour1:g}h→{hour2:g}h"
    return diff


# ── Build the full list of consecutive-pair diffs: (1,2), (2,3), (3,4), ... ──
step_diffs = []
for (hour1, df1), (hour2, df2) in zip(checkpoints[:-1], checkpoints[1:]):
    step_diffs.append(consecutive_diff(hour1, df1, hour2, df2))

all_step_diffs = pd.concat(step_diffs)

print(f"Computed {len(step_diffs)} consecutive-pair comparisons.")
display(all_step_diffs.groupby("step_label", sort=False).head(2))

## Cell 4 — Statistical Aggregation Per Step

For each consecutive step we summarise both difference columns with **mean**, **std**, and **max**
across all features. This is the table that feeds the convergence plot.

In [ ]:
def build_summary(diff_df: pd.DataFrame, hour1, hour2, perms1, perms2, label: str) -> pd.DataFrame:
    """One-row summary (mean/std/max) for a single consecutive-pair diff DataFrame."""
    stats = {"hour_from": hour1, "hour_to": hour2, "perms_from": perms1, "perms_to": perms2}
    for col, prefix in [("absolute_diff", "AbsDiff"), ("abs_absolute_diff", "AbsAbsDiff")]:
        stats[f"{prefix}_Mean"] = diff_df[col].mean()
        stats[f"{prefix}_Std"]  = diff_df[col].std()
        stats[f"{prefix}_Max"]  = diff_df[col].max()
    return pd.DataFrame(stats, index=[label])


summary_rows = []
for (hour1, df1), (hour2, df2), diff_df in zip(checkpoints[:-1], checkpoints[1:], step_diffs):
    label = f"{hour1:g}h vs {hour2:g}h"
    # Number of permutations sampled so far, stored on every row of a checkpoint
    perms1 = int(df1["num_permutations"].iloc[0])
    perms2 = int(df2["num_permutations"].iloc[0])
    summary_rows.append(build_summary(diff_df, hour1, hour2, perms1, perms2, label))

summary = pd.concat(summary_rows)
summary.index.name = "Comparison"

print("=== Step-by-step Stability Summary ===")
display(summary.style
    .format({c: "{:.6f}" for c in summary.columns if c not in ("hour_from", "hour_to", "perms_from", "perms_to")})
    .format({"hour_from": "{:.1f}", "hour_to": "{:.1f}", "perms_from": "{:d}", "perms_to": "{:d}"})
    .set_caption("Consecutive-checkpoint Shapley-value stability (i vs i+1)")
    .highlight_max(axis=0, subset=[c for c in summary.columns if "Mean" in c or "Max" in c], color="#FFDDC1")
)

## Cell 5 — Convergence Curve (Loss-style Plot)

Plots **mean** and **max** absolute difference at each step against the later checkpoint's
elapsed hour. As the Monte Carlo estimate converges, this curve should trend toward zero —
exactly like a training-loss curve flattening out.

In [ ]:
MAX_HOUR = 150   # x-axis cutoff, in hours — change this one value to adjust both plots


def plot_convergence(summary: pd.DataFrame, metric_prefix: str, ylabel: str,
                      max_hour: float = MAX_HOUR, save_path: str = None) -> None:
    """
    Plot Mean and Max curves for a given metric prefix ('AbsDiff' or 'AbsAbsDiff')
    against the 'perms_to' x-axis (the number of permutations at the later
    checkpoint in each consecutive pair). Only checkpoints up to `max_hour` are
    plotted — pass a different value here (or change the MAX_HOUR constant above)
    to show more or less of the curve. If `save_path` is
    given, the figure is written there (format inferred from the extension, e.g.
    .png, .pdf, .svg).
    """
    plot_df = summary[summary["hour_to"] <= max_hour]
    x = plot_df["perms_to"].values
    mean_col = f"{metric_prefix}_Mean"
    max_col  = f"{metric_prefix}_Max"

    # constrained_layout handles large/rotated axis labels far more reliably
    # than tight_layout() — it recomputes margins on every draw (including
    # plt.show()), rather than a one-time layout pass that can leave a big
    # label clipped at the figure edge.
    fig, ax = plt.subplots(figsize=(11, 4.2), constrained_layout=True)

    ax.plot(x, plot_df[mean_col], marker="o", markersize=4, linewidth=2, color="#4C72B0",
            label="Mean |Δ| across features")
    ax.plot(x, plot_df[max_col], marker="s", markersize=4, linewidth=2, linestyle="--", color="#C44E52",
            label="Max |Δ| across features")

    # Shade ±1 std around the mean line for extra context
    std_col = f"{metric_prefix}_Std"
    ax.fill_between(
        x,
        (plot_df[mean_col] - plot_df[std_col]).clip(lower=0),
        plot_df[mean_col] + plot_df[std_col],
        color="#4C72B0", alpha=0.15, label="Mean ± 1 Std",
    )

    ax.set_xlabel("Number of Permutations", labelpad=6, fontsize=16)
    ax.set_ylabel(ylabel, labelpad=6, fontsize=16)
    ax.set_xlim(left=0, right=x.max())
    ax.xaxis.set_major_locator(mticker.MaxNLocator(nbins=5))
    ax.yaxis.set_major_locator(mticker.MaxNLocator(nbins=4))
    ax.tick_params(axis="both", labelsize=13)
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.2f"))
    ax.grid(True, linestyle="--", alpha=0.4)
    ax.legend(framealpha=0.85, fontsize=13, loc="upper right")
    apply_paper_font(fig)

    if save_path is not None:
        fig.savefig(save_path, dpi=300, bbox_inches="tight")
        print(f'✓ Saved: {save_path}')

    plt.show()


# ── Plot 1: shapley_value convergence ───────────────────────────────────────
plot_convergence(
    summary, metric_prefix="AbsDiff",
    ylabel="|Δ shapley_value|",
    save_path=OUTPUT_DIR / f"convergence_shapley_value_{DATA}_{TASK}_{MODEL}.pdf",
)

# ── Plot 2: abs_shapley_value convergence ───────────────────────────────────
plot_convergence(
    summary, metric_prefix="AbsAbsDiff",
    ylabel="|Δ abs_shapley_value|",
    save_path=OUTPUT_DIR / f"convergence_abs_shapley_value_{DATA}_{TASK}_{MODEL}.pdf",
)

## Cell 6 — Per-feature Breakdown at the Latest Step

A sorted bar chart of the most recent consecutive comparison, to spot exactly which
features are still moving the most right now.

In [ ]:
latest_diff  = step_diffs[-1]
latest_label = latest_diff["step_label"].iloc[0]

plot_df = latest_diff[["absolute_diff"]].sort_values("absolute_diff", ascending=False)

fig, ax = plt.subplots(figsize=(max(10, len(plot_df) * 0.5), 5.5))
ax.bar(plot_df.index, plot_df["absolute_diff"], color="#4C72B0", alpha=0.85, edgecolor="white")
ax.set_xticklabels(plot_df.index, rotation=45, ha="right", fontsize=9)
ax.set_ylabel("|Δ shapley_value|")
ax.set_title(f"Per-feature change, most recent step: {latest_label}", fontsize=13, fontweight="bold", pad=12)
ax.grid(axis="y", linestyle="--", alpha=0.4)
apply_paper_font(fig)
plt.tight_layout()
plt.show()

In [ ]:
# ── Per-Feature Relative Percentage Change Table (Transposed) ──────────────

# Create an empty dictionary to store the relative change of each feature per step
per_feature_rel_changes = {}

for (hour1, df1), (hour2, df2) in zip(checkpoints[:-1], checkpoints[1:]):
    shared_idx = df1.index.intersection(df2.index)
    a = df1.loc[shared_idx]
    b = df2.loc[shared_idx]
    
    # Using epsilon to avoid division by zero
    eps = 1e-7
    
    # Calculate percentage change for each feature's absolute Shapley value
    rel_abs_diff = ((b["abs_shapley_value"] - a["abs_shapley_value"]) / (a["abs_shapley_value"] + eps)).abs() * 100
    
    step_label = f"{hour1:g}h→{hour2:g}h"
    per_feature_rel_changes[step_label] = rel_abs_diff

# Build the DataFrame and transpose it (.T) so hours are rows and features are columns
df_features_rel_transposed = pd.DataFrame(per_feature_rel_changes).T

print("=== Per-Feature Relative Percentage Change (abs_shapley_value) ===")
display(df_features_rel_transposed.style
    .format("{:.2f}%")
    .set_caption("Relative percentage change across checkpoints (Rows: Time Steps, Columns: Features)")
    .background_gradient(cmap="YlOrRd", axis=None) # Highlight high percentage changes globally
)

In [ ]:
# ── Rank Stability Analysis ─────────────────────────────────────────────────

rank_summary_rows = []

for (hour1, df1), (hour2, df2) in zip(checkpoints[:-1], checkpoints[1:]):
    shared_idx = df1.index.intersection(df2.index)
    a = df1.loc[shared_idx].copy()
    b = df2.loc[shared_idx].copy()
    
    total_features = len(shared_idx)
    top_25_count = max(1, int(np.ceil(total_features * 0.25)))
    
    # Rank features based on their absolute importance (descending order)
    a["rank"] = a["abs_shapley_value"].rank(ascending=False, method="min")
    b["rank"] = b["abs_shapley_value"].rank(ascending=False, method="min")
    
    # 1. Calculations for all features
    # A feature shifted if its rank at t1 is different from its rank at t2
    changed_all = (a["rank"] != b["rank"]).sum()
    pct_changed_all = (changed_all / total_features) * 100
    
    # 2. Calculations for the Top 25% features
    # We define the Top 25% cohort using the baseline hour (hour1)
    top_25_features = a.nsmallest(top_25_count, "rank").index
    
    changed_top25 = (a.loc[top_25_features, "rank"] != b.loc[top_25_features, "rank"]).sum()
    pct_changed_top25 = (changed_top25 / top_25_count) * 100
    
    label = f"{hour1:g}h vs {hour2:g}h"
    rank_summary_rows.append({
        "Comparison": label,
        "Total Features": total_features,
        "All: Shifted Count": changed_all,
        "All: Shifted (%)": pct_changed_all,
        f"Top 25% (n={top_25_count}): Shifted Count": changed_top25,
        "Top 25%: Shifted (%)": pct_changed_top25
    })

df_rank_summary = pd.DataFrame(rank_summary_rows).set_index("Comparison")

# ── Display Styled Summary Table ────────────────────────────────────────────
print("=== Feature Rank Displacement Summary ===")
display(df_rank_summary.style
    .format({
        "All: Shifted (%)": "{:.2f}%",
        "Top 25%: Shifted (%)": "{:.2f}%",
        "Total Features": "{:d}",
        "All: Shifted Count": "{:d}",
        f"Top 25% (n={top_25_count}): Shifted Count": "{:d}"
    })
    .set_caption("Percentage and count of features changing their exact rank between iterations")
    .background_gradient(cmap="Blues", subset=["All: Shifted (%)", "Top 25%: Shifted (%)"])
)

## Cell 7 — (Optional) Export Summary to CSV

Re-run the whole notebook later (as more checkpoints land) to refresh this file.

In [ ]:
out_path = OUTPUT_DIR / "monte_carlo_convergence_summary.csv"
summary.to_csv(out_path)
print(f"Summary saved to: {out_path}")